# Drone RL with LLM-Guided Curriculum Learning

Minimal foundation notebook for the final project report and quick manual smoke checks.


## Project overview

This project studies reinforcement learning for single-drone trajectory tracking in PyBullet, with deterministic task validation and an eventual LLM-guided curriculum generator. The notebook is the final report surface: it should explain the workflow, demonstrate the reusable modules in `src/`, and avoid expensive training by default.

Default notebook posture:
- `TRAIN_FROM_SCRATCH = False`
- `RUN_QUICK_DEMO = True`
- `USE_SAVED_RESULTS = True`


## Research question

Can an LLM propose valid and useful curriculum tasks that improve drone RL training compared with direct training on hard tasks or a fixed manual curriculum?


## Current foundation status

The current foundation checks focus on the reusable project packages and fast smoke paths:
- package alias imports from `src`
- external storage path resolution without creating directories
- smoke configuration loading from `configs/smoke/trajectory_validation.yaml`
- deterministic hover and circle trajectory generation
- deterministic trajectory task validation
- one headless PyBullet environment reset and step with a zero action

These cells do not train PPO, call an LLM, or write storage artifacts.


## Imports and package aliases


In [ ]:
from src import envs, experiments, trajectories, utils, validation

TRAIN_FROM_SCRATCH = False
RUN_QUICK_DEMO = True
USE_SAVED_RESULTS = True

for package_name, package in (
    ("envs", envs),
    ("experiments", experiments),
    ("trajectories", trajectories),
    ("utils", utils),
    ("validation", validation),
):
    print(f"{package_name}.__all__ = {package.__all__}")

## Storage/path check


In [ ]:
project_root = utils.paths.get_project_root()
storage_root = utils.paths.get_storage_root()
results_root = utils.paths.get_results_root()
models_root = utils.paths.get_models_root()
tmp_root = utils.paths.get_tmp_root()

for label, resolved_path in (
    ("project root", project_root),
    ("storage root", storage_root),
    ("results root", results_root),
    ("models root", models_root),
    ("tmp root", tmp_root),
):
    print(f"{label}: {resolved_path}")

## Config loading check


In [ ]:
config_path = project_root / "configs/smoke/trajectory_validation.yaml"
config = experiments.config.load_experiment_config(config_path)
smoke_tasks = list(config["tasks"])

print(f"config: {config_path.relative_to(project_root)}")
print(f"name: {config['name']}")
print(f"seed: {config['seed']}")
print(f"tasks: {len(smoke_tasks)}")

## Trajectory generation check


In [ ]:
hover_task = next(task for task in smoke_tasks if task["shape"] == validation.contracts.SHAPE_HOVER)
circle_task = next(task for task in smoke_tasks if task["shape"] == validation.contracts.SHAPE_CIRCLE)

hover_trajectory = trajectories.primitives.make_hover_trajectory(
    position=hover_task["position"],
    duration_sec=hover_task["duration_sec"],
    sample_rate_hz=hover_task["sample_rate_hz"],
)
circle_trajectory = trajectories.primitives.make_circle_trajectory(
    radius=circle_task["radius"],
    height=circle_task["height"],
    duration_sec=circle_task["duration_sec"],
    sample_rate_hz=circle_task["sample_rate_hz"],
    center=circle_task["center"],
    clockwise=circle_task["clockwise"],
)

for name, trajectory in (
    ("hover", hover_trajectory),
    ("circle", circle_trajectory),
):
    print(f"{name}: times={trajectory.times.shape}, positions={trajectory.positions.shape}")

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(5, 5))
ax.plot(circle_trajectory.positions[:, 0], circle_trajectory.positions[:, 1], label="circle")
ax.scatter(
    hover_trajectory.positions[0, 0],
    hover_trajectory.positions[0, 1],
    label="hover",
    zorder=3,
)
ax.set_aspect("equal", adjustable="box")
ax.set_xlabel("x [m]")
ax.set_ylabel("y [m]")
ax.set_title("Smoke trajectories")
ax.legend()
plt.show()

## Task validation check


In [ ]:
limits = validation.tasks.ValidationLimits(**config["validation_limits"])

for name, trajectory in (
    ("hover trajectory", hover_trajectory),
    ("circle trajectory", circle_trajectory),
):
    result = validation.tasks.validate_trajectory(trajectory=trajectory, limits=limits)
    messages = "; ".join(result.messages) if result.messages else "ok"
    print(f"{name}: valid={result.is_valid}, messages={messages}")

for task in smoke_tasks:
    result = validation.tasks.validate_task(task=task, limits=limits)
    messages = "; ".join(result.messages) if result.messages else "ok"
    print(f"{task['shape']} task: valid={result.is_valid}, messages={messages}")

## PyBullet drone environment smoke check


In [ ]:
import numpy as np
from gymnasium.spaces import Box

GYMNASIUM_STEP_RESULT_LENGTH = 5

smoke_env = envs.builders.make_hover_aviary_env(gui=False, record=False)
try:
    observation, _ = smoke_env.reset(seed=int(config["seed"]))
    action_space = smoke_env.action_space
    if not isinstance(action_space, Box):
        msg = f"expected Box action space, got {type(action_space).__name__}"
        raise TypeError(msg)

    zero_action = np.zeros(action_space.shape, dtype=action_space.dtype)
    step_result = smoke_env.step(zero_action)
    if len(step_result) == GYMNASIUM_STEP_RESULT_LENGTH:
        next_observation, reward, terminated, truncated, _ = step_result
    else:
        next_observation, reward, done, _ = step_result
        terminated = bool(done)
        truncated = False

    observation_array = np.asarray(next_observation, dtype=float)
    print(f"finite observation: {np.all(np.isfinite(observation_array))}")
    print(f"reward: {float(reward):.6f}")
    print(f"terminated: {bool(terminated)}")
    print(f"truncated: {bool(truncated)}")
    print(f"action shape: {zero_action.shape}")
    print(f"observation shape: {observation_array.shape}")
finally:
    smoke_env.close()

## Next development steps

1. Expand trajectory primitives and validation coverage for the planned curriculum task shapes.
2. Add environment wrappers for trajectory tracking observations, rewards, and episode termination.
3. Implement non-training evaluation utilities for saved runs and lightweight notebook visualizations.
4. Add the LLM curriculum proposal schema and deterministic validation feedback loop.
5. Later, connect saved training results and comparison plots without requiring full retraining in the notebook.
